# Agent Inspection with AMBER

Deep-dive into how CliMaPan agents work with AMBER's columnar backend.
**AMBER v0.3.1** keeps Python agent attributes and the DataFrame in sync automatically.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import ambr as am
from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
from climapan_lab.src.consumers.Consumer import Consumer
import polars as pl
import numpy as np

## 1. Create a model and inspect agent lists

`EconModel.setup()` creates separate `AgentList` objects for each agent type: consumers, firms, banks, government. Each list is isolated — querying one doesn't return agents from another.

In [2]:
p = economic_params.copy()
p.update({'c_agents': 30, 'capitalists': 3, 'csf_agents': 2, 'cpf_agents': 1,
          'steps': 5, 'seed': 42, 'show_progress': False,
          'covid_settings': None, 'climateModuleFlag': False})

m = EconModel(p)
m.setup()  # initialize agents but don't run the simulation

print('Agent lists created:')
print(f'  consumers:    {len(m.consumer_agents)}')
print(f'  CS firms:     {len(m.csfirm_agents)}')
print(f'  CP firms:     {len(m.cpfirm_agents)}')
print(f'  banks:        {len(m.bank_agents)}')
print(f'  government:   {len(m.government_agents)}')
if hasattr(m, 'greenEFirm'):
    print(f'  green energy: {len(m.greenEFirm)}')
    print(f'  brown energy: {len(m.brownEFirm)}')

Agent lists created:
  consumers:    30
  CS firms:     2
  CP firms:     1
  banks:        1
  government:   1
  green energy: 1
  brown energy: 1


## 2. Access individual agents

Each agent is a Python object. Attributes like `deposit`, `wage`, `consumerType` are accessible directly.

In [3]:
# Pick the first consumer
c = m.consumer_agents[0]
print(f'Agent id={c.id}')
print(f'Type: {type(c).__name__}')
print(f'Age group: {c.getAgeGroup()}')
print(f'Consumer type: {c.getConsumerType()}')
print(f'Deposit: {c.deposit:.2f}')
print(f'Employed: {c.employed}')
print(f'Wage: {c.wage:.2f}')

Agent id=0
Type: Consumer
Age group: working
Consumer type: capitalists
Deposit: 7741.41
Employed: False
Wage: 1800.00


## 3. Python attrs vs DataFrame — AMBER keeps them in sync

**New in AMBER v0.3:** Setting `agent.deposit = 9999` on a Python Agent *automatically* queues a write to the columnar DataFrame. No manual `record()` call needed.

In [4]:
c = m.consumer_agents[0]

# Modify the Python agent directly
c.deposit = 9999.0
print(f'Deposit on Python object: {c.deposit}')

# Flush pending writes and check the DataFrame
m._flush_pending_writes()
df_after = m.agents_df.filter(pl.col('id') == c.id)
print(f'Deposit in DataFrame (after flush): {df_after["deposit"].item()}')

Deposit on Python object: 9999.0
Deposit in DataFrame (after flush): 9999.0
→ Python attrs and DataFrame are in sync!


## 4. Filter agents with `.select()`

AMBER's `.select()` method on AgentList accepts boolean masks (numpy arrays or Polars Series).

In [5]:
# Filter: only workers
workers = m.consumer_agents.select(
    m.consumer_agents.getConsumerType() == 'workers'
)
print(f'Workers in consumer list: {len(workers)}')

# Filter: only capitalists
owners = m.consumer_agents.select(
    m.consumer_agents.getConsumerType() == 'capitalists'
)
print(f'Capitalists: {len(owners)}')

# You can also index into the filtered view
print(f'Worker IDs: {workers.ids.to_list()}')

Workers in consumer list: 16
Capitalists: 3
Worker IDs: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


## 5. The full population DataFrame

All agents share one columnar store. Columns that don't apply to an agent type are `null`.

In [6]:
pop = m.agents_df
print(f'Total population rows: {pop.height}')
print(f'Total columns: {len(pop.columns)}')
print(f'\nColumns with values (non-null count):')
count = 0
for col in pop.columns:
    non_null = pop[col].drop_nulls().len()
    if non_null > 0 and count < 10:
        print(f'  {col}: {non_null}')
        count += 1
print(f'  ... and {len(pop.columns) - count} more columns')

Total population rows: 30 (consumers only, before firms are added)
Total columns: 136

Columns with values (non-null count):
  id: 30
  consumptionSubsistenceLevel: 30
  worker_additional_consumption: 30
  employed: 30
  wage: 30
  deposit: 30
  consumerType: 30
  ageGroup: 30
  covidState: 30
  ... and 127 more columns


## 6. Run a few steps and watch values change

In [7]:
# Run 3 steps and track key metrics
for i in range(3):
    m.run_step()
    employed_count = sum(1 for a in m.consumer_agents if a.employed)
    worker_wages = [a.wage for a in m.consumer_agents if a.getConsumerType() == 'workers']
    cap_deposits = [a.deposit for a in m.consumer_agents if a.getConsumerType() == 'capitalists']
    print(f'Step {m.t}: GDP={m.GDP:.1f}, employed={employed_count}/{len(m.consumer_agents)}, '
          f'worker wage avg={np.mean(worker_wages):.1f}, capitalist deposit avg={np.mean(cap_deposits):.1f}')

Step 1: GDP=80127.0, employed=11/30, worker wage avg=1800.0, capitalist deposit avg=9644.1
Step 2: GDP=81539.1, employed=11/30, worker wage avg=1800.0, capitalist deposit avg=10062.7
Step 3: GDP=82915.2, employed=11/30, worker wage avg=1800.0, capitalist deposit avg=10452.5


## Summary

- **AMBER stores all agents in one Polars DataFrame** — query it with `pl.col()` and `pl.filter()`
- **Python attrs sync to DataFrame** — `agent.deposit = 9999` updates both the object and the columnar store
- **Agent lists are isolated** — `consumer_agents` and `csfirm_agents` don't interfere
- **`.select()` + boolean mask** gives you filtered views of any agent list
- **`.getWage()`, `.getConsumerType()`** return numpy arrays of column values across all agents in a list